In [10]:
import requests
import json
import pandas as pd
import time
import folium

from datetime import datetime

In [2]:
# PEYTON RUN
filepath = "/Users/peyton/Desktop/TicketmasterAPIKey.txt"

In [ ]:
# KATRIEL RUN
filepath = "/Users/katriellayam/Downloads/ticketmaster_project.txt"

In [3]:
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key(filepath)

type(key)

str

In [6]:
url = f"https://app.ticketmaster.com/discovery/v2/venues.json?apikey={key}"

response = requests.get(url)

response.raise_for_status()

test_data = response.json()

test_data

{'_embedded': {'venues': [{'name': 'Sphere',
    'type': 'venue',
    'id': 'KovZ917Atbr',
    'test': False,
    'url': 'https://www.ticketmaster.com/sphere-tickets-las-vegas/venue/189524',
    'locale': 'en-us',
    'images': [{'ratio': '16_9',
      'url': 'https://s1.ticketm.net/dbimages/22784v.jpg',
      'width': 640,
      'height': 360,
      'fallback': False}],
    'postalCode': '89169',
    'timezone': 'America/Los_Angeles',
    'city': {'name': 'Las Vegas'},
    'state': {'name': 'Nevada', 'stateCode': 'NV'},
    'country': {'name': 'United States Of America', 'countryCode': 'US'},
    'address': {'line1': '255 Sands Avenue'},
    'location': {'longitude': '-115.16428960', 'latitude': '36.12072670'},
    'markets': [{'name': 'Las Vegas', 'id': '14'}],
    'dmas': [{'id': 319}],
    'upcomingEvents': {'archtics': 791,
     'ticketmaster': 862,
     '_total': 1653,
     '_filtered': 0},
    '_links': {'self': {'href': '/discovery/v2/venues/KovZ917Atbr?locale=en-us'}}},
   {'n

In [7]:
venues = test_data['_embedded']['venues']

df = pd.DataFrame([{
    'id': v.get('id'),
    'name': v.get('name'),
    'address': v.get('address', {}).get('line1'),
    'city': v.get('city', {}).get('name'),
    'state': v.get('state', {}).get('stateCode'),
    'postal_code': v.get('postalCode'),
    'timezone': v.get('timezone'),
    'latitude': v.get('location', {}).get('latitude'),
    'longitude': v.get('location', {}).get('longitude'),
    'upcoming_events_total': v.get('upcomingEvents', {}).get('_total'),
    'url': v.get('url'),
} for v in venues])

df

,id,name,address,city,state,postal_code,timezone,latitude,longitude,upcoming_events_total,url
0,KovZ917Atbr,Sphere,255 Sands Avenue,Las Vegas,NV,89169,America/Los_Angeles,36.12072670,-115.16428960,1653,https://www.ticketmaster.com/sphere-tickets-la...
1,KovZpZAFnlEA,Augusta National Golf Course,2505 Washington Rd.,Augusta,GA,30904,America/New_York,33.50620910,-82.01873250,7,https://www.ticketmaster.com/augusta-national-...
2,KovZpZA7AAEA,Madison Square Garden,7th Ave & 32nd Street,New York,NY,10001,America/New_York,40.74970620,-73.99160060,824,https://www.ticketmaster.com/madison-square-ga...
3,KovZpa3hje,Bank of America Stadium,800 South Mint Street,Charlotte,NC,28202,America/New_York,35.225789,-80.852829,295,https://www.ticketmaster.com/bank-of-america-s...
4,KovZpa2M7e,United Center,1901 W Madison,Chicago,IL,60612,America/Chicago,41.88124412,-87.67427375,402,https://www.ticketmaster.com/united-center-tic...
5,KovZ917Ax7n,Allegiant Stadium,3333 Al Davis Way,Las Vegas,NV,89118,America/Los_Angeles,36.09090000,-115.18330000,356,https://www.ticketmaster.com/allegiant-stadium...
6,KovZpakS7e,MetLife Stadium,One MetLife Stadium Drive,East Rutherford,NJ,07073,America/New_York,40.81359,-74.074493,236,https://www.ticketmaster.com/metlife-stadium-t...
7,KovZpZAEkn6A,Kia Forum,3900 W Manchester Blvd.,Inglewood,CA,90305,America/Los_Angeles,33.95830000,-118.34186800,101,https://www.ticketmaster.com/kia-forum-tickets...
8,KovZ917AtP3,Barclays Center,620 Atlantic Ave,Brooklyn,NY,11217,America/New_York,40.68285000,-73.97519000,228,https://www.ticketmaster.com/barclays-center-t...
9,KovZpaoUze,Bryce Jordan Center,127 Bryce Jordan Center,University Park,PA,16802,America/New_York,40.80864350,-77.85584480,45,https://www.ticketmaster.com/bryce-jordan-cent...


In [11]:
m = folium.Map(location=[38, -95], zoom_start=4)

for _, row in df.iterrows():
    if row["latitude"] is None or row["longitude"] is None:
        continue

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['address']}<br>"
        f"{row['city']}, {row['state']} {row['postal_code']}<br>"
        f"Upcoming Events: {row['upcoming_events_total']}"
    )

    folium.Marker(
        location=[float(row["latitude"]), float(row["longitude"])],
        popup=folium.Popup(popup_text, max_width=200),
        icon=folium.Icon(color="darkblue", icon="ticket", prefix="fa")
    ).add_to(m)

m.save("venues.html")
m

In [12]:
# pull events by month
all_events = []

months = [
    ("2025-01-01", "2025-01-31"),
    ("2025-02-01", "2025-02-28"),
    ("2025-03-01", "2025-03-31"),
    ("2025-04-01", "2025-04-30"),
    ("2025-05-01", "2025-05-31"),
    ("2025-06-01", "2025-06-30"),
    ("2025-07-01", "2025-07-31"),
    ("2025-08-01", "2025-08-31"),
    ("2025-09-01", "2025-09-30"),
    ("2025-10-01", "2025-10-31"),
    ("2025-11-01", "2025-11-30"),
    ("2025-12-01", "2025-12-31"),
]

for start, end in months:
    url = (
        f"https://app.ticketmaster.com/discovery/v2/events.json"
        f"?apikey={key}"
        f"&startDateTime={start}T00:00:00Z"
        f"&endDateTime={end}T23:59:59Z"
        f"&countryCode=US"
        f"&size=200"
    )
    response = requests.get(url)
    data = response.json()
    events = data.get("_embedded", {}).get("events", [])
    for e in events:
        venue = e.get("_embedded", {}).get("venues", [{}])[0]
        loc = venue.get("location", {})
        all_events.append({
            "name": e.get("name"),
            "date": e.get("dates", {}).get("start", {}).get("localDate"),
            "venue": venue.get("name"),
            "city": venue.get("city", {}).get("name"),
            "state": venue.get("state", {}).get("stateCode"),
            "latitude": loc.get("latitude"),
            "longitude": loc.get("longitude"),
            "url": e.get("url"),
            "month": start[:7],  # e.g. "2025-01"
        })

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"] = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

In [13]:
# map of events with month toggle control
from folium.plugins import GroupedLayerControl

m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2025-01": "January", "2025-02": "February", "2025-03": "March",
    "2025-04": "April",   "2025-05": "May",       "2025-06": "June",
    "2025-07": "July",    "2025-08": "August",    "2025-09": "September",
    "2025-10": "October", "2025-11": "November",  "2025-12": "December",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in events_df.iterrows():
    month = row["month"]
    if month not in groups:
        continue
    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m